# Docker Swarm

## Swarm mode 101
<https://docs.docker.com/engine/swarm/>

:::: {.columns}

::: {.fragment .column width="50%"}
- Swarm mode is an advanced feature for managing a cluster of Docker daemons.
- Use Swarm mode if you intend to use Swarm as a production runtime environment.

:::

::: {.fragment .column width="50%"}
![](https://raw.githubusercontent.com/docker-library/docs/471fa6e4cb58062ccbf91afc111980f9c7004981/swarm/logo.png)
:::

::::

### Create a swarm cluster

:::: {.columns}

::: {.fragment .column width="50%"}
**Init the swarm on the host (manager)**
```bash
docker swarm init
docker node ls
```
:::

::: {.fragment .column width="50%"}
**Join worker nodes**
```bash
# Get the join token and manager IP
JOIN=$(docker swarm join-token -q worker)
MANAGER_IP=$(hostname -I | awk '{print $1}')

# Run on each worker node
docker swarm join --token $JOIN \
  $MANAGER_IP:2377
```
:::

::::

### Create a service

```bash
# Create a service
docker service create --name web -p 80:80 nginx
# See it's running
docker service ps web

# Test it using curl or browser
```

### Swarm Visualizer

Deploy another service: visualizer on the manager open `http://localhost:8080`

:::: {.columns}

::: {.fragment .column width="60%"}
```bash
docker service create \
  --name=viz \
  --publish=8080:8080/tcp \
  --constraint=node.role==manager \
  --mount=type=bind,\
src=/var/run/docker.sock,\
dst=/var/run/docker.sock \
  dockersamples/visualizer
```
:::

::: {.fragment .column width="40%"}
- Shows each node as a column
- Containers appear as boxes inside each node
- Updates in real time as you scale, kill nodes, or do rolling updates
:::

::::

### Scale Up 

Let's use <https://canonical.com/multipass> to scale up locally


### Add a node - Manual setup

Launch a plain Ubuntu VM, then install Docker manually inside it:

:::: {.columns}

::: {.fragment .column width="50%"}
**Launch the VM and install Docker**
```bash
multipass launch --name worker1 \
  --cpus 1 --memory 1G --disk 5G

multipass exec worker1 -- bash -c \
  "curl -fsSL https://get.docker.com | sh && \
   sudo usermod -aG docker ubuntu"
```
:::

::: {.fragment .column width="50%"}
**Join the swarm from the host**
```bash
JOIN=$(docker swarm join-token -q worker)
MANAGER_IP=$(hostname -I | awk '{print $1}')


multipass exec worker1 -- \

  docker swarm join --token $JOIN \

  $MANAGER_IP:2377

docker node ls  
```
:::
::::


### Add a node - Cloud-init setup

Define a `cloud-init.yaml` once and let the VM provision itself automatically:

:::: {.columns}

::: {.fragment .column width="50%"}
**cloud-init.yaml** + launch
```yaml
packages:
  - docker.io
runcmd:
  - usermod -aG docker ubuntu
  - systemctl enable docker
  - systemctl start docker
```
```bash
multipass launch --name worker2 \
  --cpus 1 --memory 1G --disk 5G \
  --cloud-init cloud-init.yaml
```
:::

::: {.fragment .column width="50%"}
**Join the swarm from the host**
```bash
JOIN=$(docker swarm join-token -q worker)
MANAGER_IP=$(hostname -I | awk '{print $1}')


multipass exec worker2 -- \

  docker swarm join --token $JOIN \

  $MANAGER_IP:2377

docker node ls
```
:::

::::

#### Focus Cloud Init

<https://cloud-init.io/>

- Cloud-init is the industry standard multi-distribution method for cross-platform cloud instance initialization. It is supported across all major public cloud providers, provisioning systems for private cloud infrastructure, and bare-metal installations.

- During boot, cloud-init identifies the cloud it is running on and initializes the system accordingly. Cloud instances will automatically be provisioned during first boot with networking, storage, SSH keys, packages and various other system aspects already configured.

- Cloud-init provides the necessary glue between launching a cloud instance and connecting to it so that it works as expected.

### Scale a service

:::: {.columns}

::: {.fragment .column width="50%"}
Scale the `web` service to 3 replicas 

Swarm will spread them across manager + workers:
```bash
docker service scale web=3
```
:::

::: {.fragment .column width="50%"}
Verify the replicas and which node each runs on:
```bash
docker service ps web
```
Kill a worker and watch Swarm reschedule the replica automatically:
```bash
multipass stop worker1
docker service ps web
```
:::

::::

### Cleanup

:::: {.columns}

::: {.fragment .column width="50%"}
**Remove services and leave the swarm (host)**
```bash
# Remove all services
docker service rm viz web

# Leave and dismantle the swarm
docker swarm leave --force
```
:::

::: {.fragment .column width="50%"}
**Delete the worker VMs**
```bash
# Stop and delete workers
multipass stop worker1 worker2
multipass delete worker1 worker2
multipass purge
```
:::

::::

## Swarm + Docker Compose = Stack 

<https://docs.docker.com/engine/swarm/stack-deploy/>

When running Docker Engine in swarm mode, you can use `docker stack deploy` to deploy a complete application stack to the swarm. The deploy command accepts a stack description in the form of a Compose file.

### Setup the cluster

:::: {.columns}

::: {.fragment .column width="50%"}
**Launch 2 workers and init the swarm**
```bash
# Launch workers with Docker pre-installed
multipass launch --name worker1 \
  --cpus 1 --memory 1G --disk 5G \
  --cloud-init cloud-init.yaml
multipass launch --name worker2 \
  --cpus 1 --memory 1G --disk 5G \
  --cloud-init cloud-init.yaml

# Init the swarm on the host (manager)
docker swarm init
```
:::

::: {.fragment .column width="50%"}
**Join the workers to the swarm**
```bash
# Get the join token
JOIN=$(docker swarm join-token -q worker)
MANAGER_IP=$(hostname -I | awk '{print $1}')

# Join both workers
multipass exec worker1 -- \
  docker swarm join --token $JOIN \
  $MANAGER_IP:2377
multipass exec worker2 -- \
  docker swarm join --token $JOIN \
  $MANAGER_IP:2377

# Verify all 3 nodes are up
docker node ls
```
:::

::::

### Create a Compose file

`nginxdemos/hello` serves a page showing the **container hostname and IP**.

**docker-compose.yml**
```yaml
services:
  web:
    image: nginxdemos/hello
    ports:
      - "80:80"
    deploy:
      replicas: 3
      restart_policy:
        condition: on-failure
      update_config:
        parallelism: 1
        delay: 10s
  viz:
    image: dockersamples/visualizer
    ports:
      - "8080:8080"
    volumes:
      - /var/run/docker.sock:/var/run/docker.sock
    deploy:
      placement:
        constraints:
          - node.role == manager
```

### Run with Docker Compose (local, no swarm)

:::: {.columns}

::: {.fragment .column width="50%"}
Start the stack locally — useful for development and testing:
```bash
docker compose up -d
docker compose ps
```
Stop it:
```bash
docker compose down
```
:::

::: {.fragment .column width="50%"}
- `deploy:` keys are **ignored** by `docker compose`
- Only 1 replica runs locally
- Good for iterating on the Compose file before deploying to Swarm
:::

::::

### Deploy as a Stack on Swarm (host + 2 workers)

:::: {.columns}

::: {.fragment .column width="50%"}
Deploy the stack — Swarm spreads the 3 `web` replicas across all nodes:
```bash
docker stack deploy -c docker-compose.yml myapp
```
Inspect:
```bash
docker stack ls
docker stack services myapp
docker stack ps myapp
```
:::

::: {.fragment .column width="50%"}
Tear it down:
```bash
docker stack rm myapp
```
- `deploy:` keys are now **honoured** — 3 replicas, placement constraint, update policy all apply
- The visualizer runs only on the manager (`node.role == manager`)
- Open `http://localhost:8080` to watch all 3 `web` replicas spread across the cluster
:::

::::